In [ ]:
import minari
import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader

dataset = minari.load_dataset('Box2D/CarRacing-v3/expert-v0')
all_obs = np.vstack([e.observations for e in dataset.iterate_episodes()])
print(f"Dataset loaded. Total observations: {len(all_obs)}")
print(f"Observation shape: {all_obs[0].shape}")

all_obs_normalized = all_obs.astype(np.float32) / 255.0
all_obs_tensor = torch.from_numpy(all_obs_normalized).permute(0, 3, 1, 2)

# Create a TensorDataset and a DataLoader
batch_size = 64
train_dataset = TensorDataset(all_obs_tensor)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

print(f"Data prepared for PyTorch. Tensor shape: {all_obs_tensor.shape}")

In [ ]:
from tqdm import tqdm
from src.models.vae import VAE, vae_loss

# Hyperparameters
epochs = 20
learning_rate = 1e-3
latent_dim = 32
beta = 1.0 # Weight for the KL divergence term

# Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = VAE(latent_dim=latent_dim).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

print(f"Training on {device}...")

# Training Loop
for epoch in range(epochs):
    model.train()
    train_loss = 0
    
    # Use tqdm for a nice progress bar
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")
    
    for batch in pbar:
        # Data is a list with one element, the image tensor
        data = batch[0].to(device)
        
        # Forward pass
        recon_batch, mu, log_var = model(data)
        
        # Calculate loss
        loss = vae_loss(recon_batch, data, mu, log_var, beta)
        
        # Backward pass and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        pbar.set_postfix({'loss': loss.item() / len(data)})

    avg_loss = train_loss / len(train_loader.dataset)
    print(f"====> Epoch: {epoch+1} Average loss: {avg_loss:.4f}")

# Save the trained model
torch.save(model.state_dict(), 'carracing_vae.pth')
print("Model training complete and saved to carracing_vae.pth")

In [ ]:
# 1. Load your trained model
model = VAE(latent_dim=latent_dim)
model.load_state_dict(torch.load('carracing_vae.pth'))
model.to(device)
model.eval() # Set the model to evaluation mode

# 2. Get a single observation and preprocess it
sample_obs = all_obs[42] # Get any observation
image = torch.from_numpy(sample_obs.astype(np.float32) / 255.0)
image = image.permute(2, 0, 1).unsqueeze(0).to(device) # (H,W,C) -> (C,H,W) -> (1,C,H,W)

# 3. Encode the image
with torch.no_grad(): # No need to calculate gradients
    mu, log_var = model.encode(image)

# The mean 'mu' is typically used as the final compressed representation
encoded_vector = mu.cpu().numpy().flatten()

print(f"Original image shape: {sample_obs.shape}")
print(f"Encoded vector shape: {encoded_vector.shape}")
print(f"Encoded vector (mu): \n{encoded_vector}")